## Tiresias Export - Περίληψη Διαδικασίας

Το notebook αυτό παράγει τα αρχεία **Announcement** και **Acceptance** που αποστέλλονται στον Τειρεσία κατά τα migrations (onboarding/offboarding/εσωτερικό).

**Βήματα:**
1. **Επιλογή τύπου αρχείου** ανάλογα με το migration:
   - Onboarding → μόνο Acceptance
   - Offboarding → μόνο Announcement
   - Εσωτερικό migration → και τα δύο
2. **Συλλογή περιμέτρου** δανείων από `p-cdh.derivative.dexposure` & `d-cdh.reference.tiresias_ast_perim` (αφαίρεση prefix)
3. **Δημιουργία τελικής περιμέτρου** (view `Perimeter`) με join στους πίνακες `tiresias.export_business` / `export_consumer`
4. **Έλεγχοι** για δάνεια που λείπουν (flags, loan/customer tables, subsidy exclusions)
5. **Παραγωγή Announcement File** → view `AnnouncementFileExp`, με χειροκίνητη ρύθμιση πεδίων (ημερομηνία μεταβίβασης, κωδικοί ΕΔΑΔΠ/Fund)
6. **Παραγωγή Acceptance File** → CONCAT query βάσει `Perimeter`

**Ονοματολογία αρχείων:**
- Business (MME): `Announcement_MME_<bank>_<portfolio>_<date>.txt` / `AcceptanceFile_MME_...`
- Consumer (SSX): `Announcement_SSX_<bank>_<portfolio>_<date>.txt` / `AcceptanceFile_SSX_...`

**Βασικοί πίνακες:** `d-cdh.reference.tiresias_portfolio`, `d-cdh.reference.tiresias_ast_perim`, `d-cdh-delivery.tiresias.export_business/consumer`, `d-cdh-delivery.tiresias.loan/loan_customer/customer`

Η συγκεκριμένη διαδικασία τρέχει απο κάποια  scripts  που θα βρείτε στα επισυναπτώμενα όποτε γίνεται κάποιο migrations(onboarding/offboarding/ εσωτερικό migration) ώστε να σταλούν στον tiresias τα αντίστοιχα για την κάθε περιπτωση Announcement και Acceptance Files.

Απο τα scripts αυτα διαλέγουμε με βάση τον τύπο του migration που γίνεται , η αντιστοιχία ειναι όπως φαίνεται παρακάτω : 

Σε περίπτωση Onboarding( όταν έρχονται καινουρια δάνεια μονο) → Στέλνουμε μόνο Acceptance files

Σε περίπτωση Offboarding( όταν φευγουν δάνεια μονο) → Στέλνουμε μόνο Announcement files

Σε περίπτωση εσωτερικού migration ( όταν αλλάζει portfolio εντος Cepal) → Στέλνουμε και τα δυο αρχεία.

Σημείωση : Για τον κάθε τύπο αρχείου έχουμε 2 διαφορετικα αρχεία με βάση αν το δάνειο είναι Business και Consumer.

###Οι διαφορες στο βασικό script μεταξύ των περιπτώσεων είναι ότι στην περίπτωση καινουριου portfolio (onboarding + εσωτερικό migration) κάνουμε insert

### Perimeter

Στη συνέχεια μαζεύουμε την περίμετρο  των δανέιων για τα οποία θέλουμε να παραχθεί το αρχείο  βάσει των πινάκων του DWH και “κοβουμε” το  prefix διότι στους tiresias πίνακες δεν υπάρχει.

In [0]:
%sql
select  AgreementKey , cdi , max (BusinessDate) --substring(exposurekey,4,99),exposurekey,e.AgreementKey ,p.portfolioid 
from `p-cdh`.derivative.dexposure e

-- inner join `d-cdh`.reference.tiresias_portfolio p on lower(p.PortfolioName)=lower(e.PortfolioName) and p.TiresiasCode is not null 

-- inner join `d-cdh-delivery`.tiresias.NPA a on e.ExposureKey = a.`Loan ID` and e.CDI = a.`Borrower ID`
where e.BusinessDate='2026-06-23'
--and IsActiveFlag= TRUE 

-- and lower(p.PortfolioName) like lower('%MZW%')
and AgreementKey in ('MZW208**','MZW160**','MZW3989389.','MZW2920590**','MZW10760796','MZW0010/205300000003287','MZW0010/205300000003506','MZW0010/205500000002883','MZW10760830') AND CDI in ('MZW6170391','MZW6172650','MZW5641957')
group by  AgreementKey ,cdi

In [0]:
%sql

DROP VIEW IF EXISTS ACCPerimeter;

CREATE OR REPLACE TEMPORARY VIEW ACCPerimeter AS
WITH old_portfolio AS (
  -- Equivalent of on-prem CurrentRecord=1: latest snapshot per account in old portfolio
  SELECT 
      substring(ExposureKey, 4, 99) AS AccountNumber
    , PortfolioName
    , ROW_NUMBER() OVER (PARTITION BY substring(ExposureKey, 4, 99) ORDER BY BusinessDate DESC) AS rn
  FROM `p-cdh`.derivative.dexposure
  WHERE PortfolioName <> 'MZW'
)
SELECT 
    substring(e.ExposureKey, 4, 99) AS AccountNumber
  , e.ExposureKey AS assetkey
  , e.AgreementKey AS AgreementNumber
  , p.portfolioid AS NEWPortfolioID
  , p_old.portfolioid AS OLDPortfolioID
FROM `p-cdh`.derivative.dexposure e
INNER JOIN `d-cdh`.reference.tiresias_portfolio p --allagi se p.portfolio with businessdate

    ON lower(p.PortfolioName) = lower(e.PortfolioName)
-- inner join `d-cdh-delivery`.tiresias.NPA a on e.ExposureKey = a.`Loan ID` and e.CDI = a.`Borrower ID`
-- LEFT JOIN: include accounts even if they have no old portfolio
LEFT JOIN old_portfolio e_old
    ON e_old.AccountNumber = substring(e.ExposureKey, 4, 99)
    AND e_old.rn = 1
LEFT JOIN `d-cdh`.reference.tiresias_portfolio p_old
    ON lower(p_old.PortfolioName) = lower(e_old.PortfolioName)
WHERE e.BusinessDate = '2026-06-22' -- p.AssignmentDate
    AND e.IsActiveFlag = TRUE
    AND lower(p.PortfolioName) LIKE lower('%MONZA%')
    AND e.ExposureKey LIKE 'MZW%'
    and AgreementKey in ('MZW208**','MZW160**','MZW3989389.','MZW2920590**','MZW10760796','MZW0010/205300000003287','MZW0010/205300000003506','MZW0010/205500000002883','MZW10760830') AND CDI in ('MZW6170391','MZW6172650','MZW5641957') -- einai perimetros aftou tou run
    ;

SELECT count(*) FROM ACCPerimeter;

Μετα αφου μαζέψαμε αυτή την πρώτη περίμετρο ,προχωραμε στην κανονική  και τελική περίμετρο που μαζεύει τους λογαριασμούς σε πίνακες του tiresias schema.

In [0]:
%sql
    
--create the actual perimeter reading the table export_consumer_hist or export_business_hist (latest date)
--should include debts and debtors 
--the join with history will give us whatever we send to tiresias regarding the specific debts-debtors (so join with old portfolio code )
Create or replace temporary view Perimeter as
--business
select `ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`
      ,`ΗΜ_ΤΕΛΕΥΤΑΙΑΣ_ΕΝΗΜΕΡΩΣΗΣ`
      ,`ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΔΑΝΕΙΟΥ`
      ,`ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΟΦΕΙΛΕΤΗ`
      ,`ΑΦΜ`
      ,`ΑΔΤ`
      ,`ΗΜ_ΕΚΔΟΣΗΣ_ΑΔΤ`
      , substring(coalesce(`ΚΩΔΙΚΟΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗΣ` , a.agreementNumber),1,15) AgrementNumberTir
      ,x.tiresiasCode NEWPortfolioCode      --update accordigly 
                 ,x.alias NEWPortfolioName   --update accordigly 
                 ,z.tiresiasCode OLDPortfolioCode
      ,a.agreementNumber
                 ,'B' fileType --(B=business / C=consumer)
                 ,'M' InfoType --Τύπος Πληροφορίας D = Δάνειο C = Κάρτα M (Λατινικό) = Επιχειρηματική χορήγηση
               -- ,H.*

from ACCPerimeter A
join `d-cdh`.reference.tiresias_portfolio x on x.portfolioid=a.newportfolioid
left join `d-cdh`.reference.tiresias_portfolio z on z.portfolioid=a.OldPortfolioID
join `d-cdh-delivery`.tiresias.export_business H on h.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`=a.AccountNumber --and h.portfolio=z.alias
--select max(rpt_date) from pltfrm.tiresias.export_business_hist

union all

--consumer
select `ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`
      ,`ΗΜ_ΤΕΛΕΥΤΑΙΑΣ_ΕΝΗΜΕΡΩΣΗΣ`
      ,`ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΔΑΝΕΙΟΥ`
      ,`ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΟΦΕΙΛΕΤΗ`
      ,`ΑΦΜ`
      ,`ΑΔΤ`
      ,`ΗΜ_ΕΚΔΟΣΗΣ_ΑΔΤ`
      , substring(coalesce(`ΚΩΔΙΚΟΣ_ΚΟΙΝΗΣ_ΣΥΝΔΕΔΕΜΕΝΗΣ_ΡΥΘΜΙΣΗΣ` , a.agreementNumber),1,15) AgrementNumberTir
      ,x.tiresiasCode NEWPortfolioCode      --update accordigly 
                 ,x.alias NEWPortfolioName   --update accordigly 
                 ,z.tiresiasCode OLDPortfolioCode
      ,a.agreementNumber
                 ,'C' fileType --(B=business / C=consumer)
                 ,'D' InfoType --Τύπος Πληροφορίας D = Δάνειο C = Κάρτα M (Λατινικό) = Επιχειρηματική χορήγηση
               -- ,H.*
from ACCPerimeter  A
join `d-cdh`.reference.tiresias_portfolio x on x.portfolioid=a.newportfolioid
left join `d-cdh`.reference.tiresias_portfolio z on z.portfolioid=a.OldPortfolioID
join `d-cdh-delivery`.tiresias.export_consumer H on h.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`=a.AccountNumber --and h.portfolio=z.alias

;

In [0]:
%sql
select count(distinct `ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`) from Perimeter 

Σε αυτό το σημείο ελέγχουμε σε σχέση με την  αρχική περίμετρο  αν μας λείπουν δάνεια. Σε αυτή την περίπτωση αν βγάλει το παρακάτω query  αποτέλεσμα πρέπει να ψάξουμε  στην υλοποίηση που “γέμισαν” οι πίνακες του τειρεσία γιατι  γίνονται excluded οι λογαριασμοί.

In [0]:
%sql
select distinct AccountNumber 
from ACCPerimeter

In [0]:
%sql

--check if all perimeter comes in files
select distinct AccountNumber 
from ACCPerimeter
except 

select distinct `ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`

from Perimeter

Σε περίπτωση που δεν βρουμε λογαριασμούς(καθόλου ή κάποιους που λείπουν) είναι 3 οι πρώτοι έλεγχοι που μπορούμε να κάνουμε.

Ελέγχουμε τους πίνακες του query του #Perimeter και βλέπουμε γιατι συνήθως αν είναι νέο το portfolio πχ μπορεί να μην έχουμε βάλει το νέο portfolio στον πίνακα portfolio ή να μην έχει τρέξει ο τειρεσίας για να “έρθουν”τα νέα δάνεια(αν δεν έχουμε τα δάνεια στον  tiresias.loan).

Αν ελέγξουμε το παραπάνω και δούμε οτι λείπουν από τους tiresias.export_business/consumer τότε πάμε στο  αντίστοιχο procedure που γεμίζει τον κάθε πίνακα απο τους 2 και κοιτάζουμε  για τα δάνεια αυτά που λέιπουν στο βασικό select του insert. Συνήθως χάνονται δάνεια λόγω κάποιο flag από αυτά που έχει το query(isexportflag/isvalid)

Τέλος μπορούμε να ελέγξουμε τον πίνακα  dwh_stg.profits.LNS_EXT_SUBSIDY_HD διοτι κόβονται τα δάνεια που ωρίσκονται σε αυτόν απο το tiresias export.

###CHECKS

Συνεχίζουμε με την παραγωγή του export κάνοντας insert στον πίνακα tiresias.AnnouncementFileExp.

Το συγκεκριμένο γίνεται βάσει του #perimeter αλλα πρέπει να προσέξουμε να αλλάξουμε κάποια συγκεκριμένα πεδία με το χέρι.

Πιο συγκεκριμένα  πρέπει να βάλουμε σωστα τα πεδία [ΗΜΕΡΟΜΗΝΙΑ ΜΕΤΑΒΙΒΑΣΗΣ],[ΚΩΔΙΚΟΣ FUND/SPV ΑΠΟ] ,[ΚΩΔΙΚΟΣ ΕΔΑΔΠ ΑΠΟ],[ΚΩΔΙΚΟΣ FUND/SPV ΣΕ] ,[ΚΩΔΙΚΟΣ ΕΔΑΔΠ ΣΕ]

[ΗΜΕΡΟΜΗΝΙΑ ΜΕΤΑΒΙΒΑΣΗΣ]  βαζουμε την ημερομηνία του migration(assignment day του ref.portfolio)

[ΚΩΔΙΚΟΣ FUND/SPV ΑΠΟ] /[ΚΩΔΙΚΟΣ FUND/SPV ΣΕ] είναι ουσιαστικά το portfolio αυτο ιδανικά θα είναι συμπληρωμένο απο τα προηγουμενα βήματα (old/new portfolio)  ή και κενό αν δεν το γνωρίζουμε και μας πουν οτι ειναι οκ απο την Τειρεσίας

[ΚΩΔΙΚΟΣ ΕΔΑΔΠ ΑΠΟ] /[ΚΩΔΙΚΟΣ ΕΔΑΔΠ  ΣΕ] είναι η διαχειριστική απο την οποία έρχεται ή πάει το δάνειο.Εδω διακρίνονται 3 περιπτώσεισ.

Σε εσωτερικό migration συμπληρώνεται το 'CPA' και στα δύο εφόσον απο τη cepal εφυγε και στη cepal θα πάει.

Σε onboarding θα ειναι CPA το ΣΕ και θα πρέπει να βρουμε τον κωδικό ΑΠΟ, ενω σε offboarding βρίσκουμε τον κωδικό για το ΣΕ από το που θα πάει το δάνειο και το ΑΠΟ είναι 'CPA'.

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW AnnouncementFileExp AS

SELECT
      ROW_NUMBER() OVER (ORDER BY p.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`) AS Id

    ,'CPA'                                         AS `ΚΩΔΙΚΟΣ ΔΙΑΧΕΙΡΙΣΤΗ ΑΠΟ`
    , p.OLDPortfolioCode                           AS `ΚΩΔΙΚΟΣ FUND/SPV ΑΠΟ`
    , p.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`            AS `ΚΩΔΙΚΟΣ ΧΟΡΗΓΗΣΗΣ ΑΠΟ`
    , p.`ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΔΑΝΕΙΟΥ`                    AS `ΧΑΡΑΚΤΗΡΙΣΜΟΣ ΧΟΡΗΓΗΣΗΣ`
    , p.InfoType                                   AS `ΤΥΠΟΣ ΠΛΗΡΟΦΟΡΙΑΣ`
    , ''                                           AS `ΚΩΔΙΚΟΣ ΔΙΑΧΕΙΡΙΣΤΗ ΣΕ`  --allagi
    , 'EUP'                                        AS `ΚΩΔΙΚΟΣ FUND/SPV ΣΕ`     --allagi
    , p.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`            AS `ΚΩΔΙΚΟΣ ΧΟΡΗΓΗΣΗΣ ΣΕ`
    , ''                                           AS `ΟΝΟΜΑ ΧΑΡΤΟΦΥΛΑΚΙΟΥ`       --allagi
    , '2026-06-23'                                 AS `ΗΜΕΡΟΜΗΝΙΑ ΜΕΤΑΒΙΒΑΣΗΣ`   --allagi 
    , COALESCE(p.`ΑΦΜ`, '000000000')               AS `ΑΦΜ ΟΦΕΙΛΕΤΗ`
    , p.`ΧΑΡΑΚΤΗΡΙΣΜΟΣ_ΟΦΕΙΛΕΤΗ`                   AS `ΧΑΡΑΚΤΗΡΙΣΜΟΣ ΟΦΕΙΛΕΤΗ`
    , substring(p.agreementnumber, 4, 99)          AS `ΑΡΙΘΜΟΣ ΣΥΜΒΑΣΗΣ ΟΦΕΙΛΗΣ`
    , p.`ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ`            AS `ΑΡΙΘΜΟΣ ΛΟΓΑΡΙΑΣΜΟΥ ΟΦΕΙΛΗΣ`
    , 'Ο'                                          AS `ΥΠΑΓΩΓΗ ΣΤΟ ΝΟΜΟ 4469`
    , ''                                           AS `ΤΑΥΤΟΤΗΤΑ ΟΦΕΙΛΗΣ ΜΗΝΥΜΑΤΟΣ Γ1`
    , 'Ο'                                          AS `ΥΠΑΓΩΓΗ ΣΤΟ ΝΟΜΟ 4605`
    , ''                                           AS `ΤΑΥΤΟΤΗΤΑ ΟΦΕΙΛΗΣ ΜΗΝΥΜΑΤΟΣ Γ2`
    , 'Ο'                                          AS `ΥΠΑΓΩΓΗ ΣΤΟ ΝΟΜΟ 3869`
    , ''                                           AS `ΤΑΥΤΟΤΗΤΑ ΟΦΕΙΛΗΣ ΜΗΝΥΜΑΤΟΣ Γ3`
    , 'Ο'                                          AS `ΥΠΑΓΩΓΗ ΣΤΟ ΝΟΜΟ 4738`
    , ''                                           AS `ΤΑΥΤΟΤΗΤΑ ΟΦΕΙΛΗΣ ΜΗΝΥΜΑΤΟΣ Γ4`
    , p.fileType                                   AS dataFile

FROM Perimeter p;

In [0]:
%sql
select count(distinct `ΚΩΔΙΚΟΣ ΧΟΡΗΓΗΣΗΣ ΑΠΟ`) from AnnouncementFileExp WHERE dataFile = 'C';

In [0]:
%sql

select count(distinct `ΚΩΔΙΚΟΣ ΧΟΡΗΓΗΣΗΣ ΑΠΟ`) from AnnouncementFileExp WHERE dataFile = 'B';

BUSINESS ANNOUNCEMENT

Για offboarding και εσωτερικο migration.

Όνομα → π.χ.Announcement_MME_014_CEL_20231117.txt 

/ Αnnouncement_MME_014_XAT_20240430.txt 

(Using origin bank code ,portfolio tiresias CODE , assignment date , MME means business)

In [0]:
%sql
SELECT CONCAT(
    RIGHT(CONCAT('00000', CAST(Id AS STRING)), 6), ';',
    LEFT(CONCAT(COALESCE(`ΚΩΔΙΚΟΣ ΔΙΑΧΕΙΡΙΣΤΗ ΑΠΟ`, ''), SPACE(3)), 3), ';',
    LEFT(CONCAT(COALESCE(`ΚΩΔΙΚΟΣ FUND/SPV ΑΠΟ`, ''), SPACE(3)), 3), ';',
    LEFT(CONCAT(COALESCE(`ΚΩΔΙΚΟΣ ΧΟΡΗΓΗΣΗΣ ΑΠΟ`, ''), SPACE(16)), 16), ';',
    LEFT(CONCAT(COALESCE(`ΧΑΡΑΚΤΗΡΙΣΜΟΣ ΧΟΡΗΓΗΣΗΣ`, ''), SPACE(1)), 1), ';',
    LEFT(CONCAT(COALESCE(`ΤΥΠΟΣ ΠΛΗΡΟΦΟΡΙΑΣ`, ''), SPACE(1)), 1), ';',
    LEFT(CONCAT(COALESCE(`ΚΩΔΙΚΟΣ ΔΙΑΧΕΙΡΙΣΤΗ ΣΕ`, ''), SPACE(3)), 3), ';',
    LEFT(CONCAT(COALESCE(`ΚΩΔΙΚΟΣ FUND/SPV ΣΕ`, ''), SPACE(3)), 3), ';',
    LEFT(CONCAT(COALESCE(`ΚΩΔΙΚΟΣ ΧΟΡΗΓΗΣΗΣ ΣΕ`, ''), SPACE(16)), 16), ';',
    LEFT(CONCAT(COALESCE(`ΟΝΟΜΑ ΧΑΡΤΟΦΥΛΑΚΙΟΥ`, ''), SPACE(20)), 20), ';',
    LEFT(CONCAT(COALESCE(`ΗΜΕΡΟΜΗΝΙΑ ΜΕΤΑΒΙΒΑΣΗΣ`, ''), SPACE(8)), 8), ';',
    LEFT(CONCAT(COALESCE(`ΑΦΜ ΟΦΕΙΛΕΤΗ`, ''), SPACE(9)), 9), ';',
    LEFT(CONCAT(COALESCE(`ΧΑΡΑΚΤΗΡΙΣΜΟΣ ΟΦΕΙΛΕΤΗ`, ''), SPACE(1)), 1), ';',
    LEFT(CONCAT(COALESCE(`ΑΡΙΘΜΟΣ ΣΥΜΒΑΣΗΣ ΟΦΕΙΛΗΣ`, ''), SPACE(50)), 50), ';',
    LEFT(CONCAT(COALESCE(`ΑΡΙΘΜΟΣ ΛΟΓΑΡΙΑΣΜΟΥ ΟΦΕΙΛΗΣ`, ''), SPACE(50)), 50), ';',
    LEFT(CONCAT(COALESCE(`ΥΠΑΓΩΓΗ ΣΤΟ ΝΟΜΟ 4469`, ''), SPACE(1)), 1), ';',
    LEFT(CONCAT(COALESCE(`ΤΑΥΤΟΤΗΤΑ ΟΦΕΙΛΗΣ ΜΗΝΥΜΑΤΟΣ Γ1`, ''), SPACE(50)), 50), ';',
    LEFT(CONCAT(COALESCE(`ΥΠΑΓΩΓΗ ΣΤΟ ΝΟΜΟ 4605`, ''), SPACE(1)), 1), ';',
    LEFT(CONCAT(COALESCE(`ΤΑΥΤΟΤΗΤΑ ΟΦΕΙΛΗΣ ΜΗΝΥΜΑΤΟΣ Γ2`, ''), SPACE(50)), 50), ';',
    LEFT(CONCAT(COALESCE(`ΥΠΑΓΩΓΗ ΣΤΟ ΝΟΜΟ 3869`, ''), SPACE(1)), 1), ';',
    LEFT(CONCAT(COALESCE(`ΤΑΥΤΟΤΗΤΑ ΟΦΕΙΛΗΣ ΜΗΝΥΜΑΤΟΣ Γ3`, ''), SPACE(50)), 50), ';',
    LEFT(CONCAT(COALESCE(`ΥΠΑΓΩΓΗ ΣΤΟ ΝΟΜΟ 4738`, ''), SPACE(1)), 1), ';',
    LEFT(CONCAT(COALESCE(`ΤΑΥΤΟΤΗΤΑ ΟΦΕΙΛΗΣ ΜΗΝΥΜΑΤΟΣ Γ4`, ''), SPACE(50)), 50), ';'
) AS line
FROM AnnouncementFileExp
WHERE dataFile = 'B'


CONSUMER ANNOUNCEMENT

Για offboarding και εσωτερικο migration.

Όνομα → π.χ.Announcement_SSX_014_CEL_20231117.txt 

/ Announcement_SSX_014_XAT_20240430.txt 

(Using origin bank code ,portfolio tiresias CODE , assignment date , SSX means consumer)

In [0]:
df = spark.sql("""
SELECT CONCAT(
       RIGHT(CONCAT('00000', CAST(Id AS STRING)), 6), ';',
       LEFT(CONCAT(COALESCE(`ΚΩΔΙΚΟΣ ΔΙΑΧΕΙΡΙΣΤΗ ΑΠΟ`, ''), SPACE(3)), 3), ';',
       LEFT(CONCAT(COALESCE(`ΚΩΔΙΚΟΣ FUND/SPV ΑΠΟ`, ''), SPACE(3)), 3), ';',
       LEFT(CONCAT(COALESCE(`ΚΩΔΙΚΟΣ ΧΟΡΗΓΗΣΗΣ ΑΠΟ`, ''), SPACE(16)), 16), ';',
       LEFT(CONCAT(COALESCE(`ΧΑΡΑΚΤΗΡΙΣΜΟΣ ΧΟΡΗΓΗΣΗΣ`, ''), SPACE(1)), 1), ';',
       LEFT(CONCAT(COALESCE(`ΤΥΠΟΣ ΠΛΗΡΟΦΟΡΙΑΣ`, ''), SPACE(1)), 1), ';',
       LEFT(CONCAT(COALESCE(`ΚΩΔΙΚΟΣ ΔΙΑΧΕΙΡΙΣΤΗ ΣΕ`, ''), SPACE(3)), 3), ';',
       LEFT(CONCAT(COALESCE(`ΚΩΔΙΚΟΣ FUND/SPV ΣΕ`, ''), SPACE(3)), 3), ';',
       LEFT(CONCAT(COALESCE(`ΚΩΔΙΚΟΣ ΧΟΡΗΓΗΣΗΣ ΣΕ`, ''), SPACE(16)), 16), ';',
       LEFT(CONCAT(COALESCE(`ΟΝΟΜΑ ΧΑΡΤΟΦΥΛΑΚΙΟΥ`, ''), SPACE(20)), 20), ';',
       LEFT(CONCAT(COALESCE(`ΗΜΕΡΟΜΗΝΙΑ ΜΕΤΑΒΙΒΑΣΗΣ`, ''), SPACE(8)), 8), ';',
       LEFT(CONCAT(COALESCE(`ΑΦΜ ΟΦΕΙΛΕΤΗ`, ''), SPACE(9)), 9), ';',
       LEFT(CONCAT(COALESCE(`ΧΑΡΑΚΤΗΡΙΣΜΟΣ ΟΦΕΙΛΕΤΗ`, ''), SPACE(1)), 1), ';',
       LEFT(CONCAT(COALESCE(`ΑΡΙΘΜΟΣ ΣΥΜΒΑΣΗΣ ΟΦΕΙΛΗΣ`, ''), SPACE(50)), 50), ';',
       LEFT(CONCAT(COALESCE(`ΑΡΙΘΜΟΣ ΛΟΓΑΡΙΑΣΜΟΥ ΟΦΕΙΛΗΣ`, ''), SPACE(50)), 50), ';',
       LEFT(CONCAT(COALESCE(`ΥΠΑΓΩΓΗ ΣΤΟ ΝΟΜΟ 4469`, ''), SPACE(1)), 1), ';',
       LEFT(CONCAT(COALESCE(`ΤΑΥΤΟΤΗΤΑ ΟΦΕΙΛΗΣ ΜΗΝΥΜΑΤΟΣ Γ1`, ''), SPACE(50)), 50), ';',
       LEFT(CONCAT(COALESCE(`ΥΠΑΓΩΓΗ ΣΤΟ ΝΟΜΟ 4605`, ''), SPACE(1)), 1), ';',
       LEFT(CONCAT(COALESCE(`ΤΑΥΤΟΤΗΤΑ ΟΦΕΙΛΗΣ ΜΗΝΥΜΑΤΟΣ Γ2`, ''), SPACE(50)), 50), ';',
       LEFT(CONCAT(COALESCE(`ΥΠΑΓΩΓΗ ΣΤΟ ΝΟΜΟ 3869`, ''), SPACE(1)), 1), ';',
       LEFT(CONCAT(COALESCE(`ΤΑΥΤΟΤΗΤΑ ΟΦΕΙΛΗΣ ΜΗΝΥΜΑΤΟΣ Γ3`, ''), SPACE(50)), 50), ';',
       LEFT(CONCAT(COALESCE(`ΥΠΑΓΩΓΗ ΣΤΟ ΝΟΜΟ 4738`, ''), SPACE(1)), 1), ';',
       LEFT(CONCAT(COALESCE(`ΤΑΥΤΟΤΗΤΑ ΟΦΕΙΛΗΣ ΜΗΝΥΜΑΤΟΣ Γ4`, ''), SPACE(50)), 50), ';'
) AS line
FROM AnnouncementFileExp
WHERE dataFile = 'B'
""").filter("line IS NOT NULL")

pdf = df.toPandas()
pdf = pdf[pdf["line"].notna()]

output_path = "/Workspace/Shared/Announcement_MME_014_LAN_20260630.txt"
with open(output_path, "w", encoding="utf-8") as f:
    for line in pdf["line"]:
        f.write(line + "\n")

print(f"File written to: {output_path}")
print(f"Total lines: {len(pdf)}")

BUSINESS ACCEPTANCE

Για onboarding και εσωτερικο migration.

Όνομα → π.χ.AccepetanceFile_MME_014_SO3_20251208.txt 

/ AccepetanceFile_MME_017_IMO_20251027.txt 

(Using origin bank code ,portfolio tiresias CODE , assignment date , MME means business)

In [0]:
%sql

WITH d AS (
    SELECT DISTINCT
          `ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ` AS AccountNumber
        , OLDPortfolioCode
    FROM Perimeter
    WHERE fileType = 'B'
),
f AS (
    SELECT
          ROW_NUMBER() OVER (ORDER BY AccountNumber, OLDPortfolioCode) AS Numbering
        , AccountNumber
        , OLDPortfolioCode
    FROM d
)
SELECT CONCAT(
      LPAD(TRIM(CAST(Numbering AS STRING)), 6, '0')
    , ';', OLDPortfolioCode
    , ';', RPAD(TRIM(AccountNumber), 16, ' ')
    , ';', 'CPA'
    , ';', RPAD(TRIM(AccountNumber), 16, ' ')
    , ';'
) AS line
FROM f;

CONSUMER ACCEPTANCE

Για onboarding και εσωτερικο migration.

Όνομα → π.χ.AccepetanceFile_SSX_017_IMO_20251027.txt 

/ AccepetanceFile_SSX_014_GAI_20250610.txt 

(Using origin bank code ,portfolio tiresias CODE , assignment date , SSX means consumer)

In [0]:
%sql

WITH d AS (
    SELECT DISTINCT
          `ΚΩΔΙΚΟΣ_ΛΟΓΑΡΙΑΣΜΟΥ_ΧΟΡΗΓΗΣΗΣ` AS AccountNumber
        , OLDPortfolioCode
    FROM Perimeter
    WHERE fileType = 'C'
),
f AS (
    SELECT
          ROW_NUMBER() OVER (ORDER BY AccountNumber, OLDPortfolioCode) AS Numbering
        , AccountNumber
        , OLDPortfolioCode
    FROM d
)

SELECT CONCAT(
      LPAD(TRIM(CAST(Numbering AS STRING)), 6, '0')
    , ';', COALESCE(OLDPortfolioCode, '')
    , ';', RPAD(TRIM(AccountNumber), 16, ' ')
    , ';', 'CPA'
    , ';', RPAD(TRIM(AccountNumber), 16, ' ')
    , ';'
) AS line
FROM f;

In [0]:
review_df = spark.sql("""
    SELECT a.* 
    FROM `p-cdh-delivery`.tiresias.loan_customer A
    INNER JOIN ACCPerimeter P 
        ON a.account_number = P.AccountNumber
    WHERE a.portfolioid = P.OLDPortfolioID  -- always the old portfolio
        AND a.Tiresias_export_flg = 1
""")

display(review_df)


In [0]:
update_recalled = spark.sql("""
    MERGE INTO `p-cdh-delivery`.tiresias.loan_customer A
    USING ACCPerimeter P
        ON A.account_number = P.AccountNumber
        AND A.portfolioid = P.OLDPortfolioID  -- always the old portfolio
    WHEN MATCHED AND A.Tiresias_export_flg = 1 THEN
        UPDATE SET 
            A.Tiresias_export_flg = 2,
            A.Tiresias_not_sent_reason = 'recalled'
""")

display(update_recalled)


Τελος για τις περιπρώσεισ  offboarding και εσωτερικού  migration  τρέχουμε και άλλο ένα κομματι κώδικα που ουσιαστικά “κλεινει” τα δάνεια λόγω του οτι έφυγαν εφόσον δεν εξοφλήθηκαν ή δεν θα έχουν κάποιο άλλο status κλεισιματος.


Ενημερώνουμε τον loan_customer να κλείσει το δάνειο, με το συγκεκριμένο closing reason.
Για την περιπτωση εσωτερικου migration το reason ειναι ‘sold internally' ενω για το offboarding ‘recalled’

In [0]:
# to cell afto kanei update prod opote tha skasei logo permissions, opote gia na to treksei kapoios prepei na ftiaksoume pali thn perimetro ACCPerimeter. Exw afhsei sto prod to adhoc ntbk /Workspace/Shared/tiresias_exports/Adhoc Loan Customer UPD For Offboarding

update_recalled = spark.sql("""
    MERGE INTO `p-cdh-delivery`.tiresias.loan_customer A
    USING ACCPerimeter P
        ON A.account_number = P.AccountNumber
        AND a.portfolioname = 'MONZA' --a.portfolioid = P.OLDPortfolioID o loan customer den exei id opote eprepe na desw me prtfl, opote to ekana karfwta  -- always the old portfolio
    WHEN MATCHED AND A.Tiresias_export_flg = 1 THEN
        UPDATE SET 
            A.Tiresias_export_flg = 2,
            A.Tiresias_not_sent_reason = 'recalled'
""")

display(update_recalled)